In [ ]:
### Phase 1: Base Architecture Training (80 Epochs)

In [ ]:
import os
import glob
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import torchvision.transforms.functional as TF

# =========================================================
# 1. DATASET WITH AUGMENTATIONS
# Explains how the raw data is loaded and transformed.
# =========================================================
class KLAChallengeDataset(Dataset):
    def __init__(self, degraded_dir, clean_dir, crop_size=128):
        self.crop_size = crop_size

        # Scans the directories strictly for .npy arrays
        deg_all = sorted(glob.glob(os.path.join(degraded_dir, '*.npy')))
        cln_all = sorted(glob.glob(os.path.join(clean_dir, '*.npy')))

        # Matches noisy and clean images by their identical filenames
        cln_dict = {os.path.basename(f): f for f in cln_all}
        self.pairs = []
        for deg_path in deg_all:
            base = os.path.basename(deg_path)
            if base in cln_dict:
                self.pairs.append((deg_path, cln_dict[base]))

        assert len(self.pairs) > 0, "No valid paired arrays found! Check folder paths."

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        deg_path, cln_path = self.pairs[idx]

        # Loads the arrays and converts them to 32-bit floats for the GPU
        deg_np = np.load(deg_path).astype(np.float32)
        cln_np = np.load(cln_path).astype(np.float32)

        # Adds a channel dimension so PyTorch reads it as (1, H, W) grayscale
        deg_tensor = torch.from_numpy(deg_np).unsqueeze(0)
        cln_tensor = torch.from_numpy(cln_np).unsqueeze(0)

        # Normalization safeguard: Forces 0-255 pixel values into a 0.0-1.0 range
        if cln_tensor.max() > 2.0:
            deg_tensor = deg_tensor / 255.0
            cln_tensor = cln_tensor / 255.0

        _, h, w = deg_tensor.shape

        # PAIRED RANDOM CROPPING:
        # Grabs a random 128x128 patch from the noisy image and the corresponding
        # 256x256 patch from the clean image to teach the AI how to upscale details.
        if w > self.crop_size and h > self.crop_size:
            x = random.randint(0, w - self.crop_size)
            y = random.randint(0, h - self.crop_size)
            deg_tensor = TF.crop(deg_tensor, y, x, self.crop_size, self.crop_size)
            cln_tensor = TF.crop(cln_tensor, y * 2, x * 2, self.crop_size * 2, self.crop_size * 2)
        elif w != self.crop_size or h != self.crop_size:
            deg_tensor = TF.resize(deg_tensor, [self.crop_size, self.crop_size], antialias=True)
            cln_tensor = TF.resize(cln_tensor, [self.crop_size * 2, self.crop_size * 2], antialias=True)

        # PAIRED AUGMENTATIONS: Randomly flips the images to artificially double the dataset size
        if random.random() > 0.5:
            deg_tensor = TF.hflip(deg_tensor)
            cln_tensor = TF.hflip(cln_tensor)
        if random.random() > 0.5:
            deg_tensor = TF.vflip(deg_tensor)
            cln_tensor = TF.vflip(cln_tensor)

        # Hard limits the pixels to prevent math crashes from crazy noise spikes
        deg_tensor = torch.clamp(deg_tensor, 0.0, 1.0)
        return deg_tensor, cln_tensor

# =========================================================
# 2. ARCHITECTURE
# Explains the specific math and layers of the AI brain.
# =========================================================
class ConvBlock(nn.Module):
    # A reusable block of two Convolution layers with LeakyReLU activation.
    # We use 'reflect' padding to stop the edges of the image from turning black.
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel_size=3, padding=1, padding_mode='reflect'),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(out_c, out_c, kernel_size=3, padding=1, padding_mode='reflect'),
            nn.LeakyReLU(0.2, inplace=True)
        )
    def forward(self, x):
        return self.conv(x)

class SR_UNet_Residual(nn.Module):
    def __init__(self):
        super().__init__()
        # Encoder: Extracts shapes and features, compressing the image size while increasing channels
        self.down1 = ConvBlock(1, 64)
        self.pool1 = nn.MaxPool2d(2)
        self.down2 = ConvBlock(64, 128)
        self.pool2 = nn.MaxPool2d(2)

        # Bottleneck: The deepest part of the network handling the most complex logic
        self.bottleneck = ConvBlock(128, 256)

        # Decoder: Rebuilds the image back to a larger size using the extracted features
        self.up1 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.up_conv1 = ConvBlock(256, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.up_conv2 = ConvBlock(128, 64)

        # Final Upscale: Forces the output to strictly become 256x256
        self.sr_up = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.sr_conv = ConvBlock(32, 32)

        self.out = nn.Conv2d(32, 1, kernel_size=1)

    def forward(self, x):
        # MATHEMATICAL TRICK: Instead of making the AI guess the entire high-res image,
        # we mathematically stretch the blurry image to 256x256 first using bicubic interpolation.
        base_upscale = F.interpolate(x, scale_factor=2.0, mode='bicubic', align_corners=False)

        # The AI only focuses on finding the complex details and edges
        x1 = self.down1(x)
        x2 = self.down2(self.pool1(x1))
        b = self.bottleneck(self.pool2(x2))

        u1 = self.up_conv1(torch.cat([x2, self.up1(b)], dim=1))
        u2 = self.up_conv2(torch.cat([x1, self.up2(u1)], dim=1))

        # residual = the tiny, missing high-frequency details (sharp edges, removed noise)
        residual = self.out(self.sr_conv(self.sr_up(u2)))

        # We add the AI's details directly on top of the stretched blurry image
        return base_upscale + residual

# =========================================================
# 3. TRAINING ENGINE (80 Epochs)
# Explains the brute-force learning loop.
# =========================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Executing on: {device}")

degraded_dir = '/content/dataset/train/NoisyLR'
clean_dir = '/content/dataset/train/GT'

checkpoint_path = '/content/drive/MyDrive/KLA_DATA/sr_unet_residual.pth'
os.makedirs(os.path.dirname(checkpoint_path), exist_ok=True)

dataset = KLAChallengeDataset(degraded_dir, clean_dir, crop_size=128)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)

model = SR_UNet_Residual().to(device)
criterion = nn.L1Loss() # L1 Loss is used because it produces sharper edges than MSE Loss
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

# Automatic Mixed Precision (AMP) makes the GPU run twice as fast by using 16-bit math where safe
scaler = torch.amp.GradScaler('cuda')
num_epochs = 80

# Cosine Annealing slowly lowers the learning rate in a curve to help the model settle perfectly
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)

print("Launching base training loop (80 Epochs)...")
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for deg, cln in dataloader:
        deg, cln = deg.to(device, non_blocking=True), cln.to(device, non_blocking=True)

        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            out = model(deg)
            loss = criterion(out, cln)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()

    scheduler.step()
    avg_loss = running_loss / len(dataloader)
    current_lr = scheduler.get_last_lr()[0]
    print(f"Epoch [{epoch+1}/{num_epochs}] - L1 Loss: {avg_loss:.6f} - LR: {current_lr:.6f}")

    torch.save({
        'epoch': epoch,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scaler_state': scaler.state_dict(),
        'loss': avg_loss
    }, checkpoint_path)

print("Base training complete. Model saved to Drive.")

In [ ]:
### Phase 2: Ultimate Fine-Tuning (10 Epochs)

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
from tqdm import tqdm

# =========================================================
# 1. STRICT DATASET (No Augmentations)
# In Phase 2, we stop cropping and flipping the images.
# The AI must look at the exact full-size images to learn the final pixel accuracy.
# =========================================================
class KLA_Dataset_Finetune(Dataset):
    def __init__(self, noisy_dir, clean_dir):
        self.noisy_dir = noisy_dir
        self.clean_dir = clean_dir
        self.files = sorted([f for f in os.listdir(noisy_dir) if f.endswith('.npy')])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]

        lr = np.load(os.path.join(self.noisy_dir, fname)).astype(np.float32)
        hr = np.load(os.path.join(self.clean_dir, fname)).astype(np.float32)

        lr_tensor = torch.from_numpy(lr).unsqueeze(0)
        hr_tensor = torch.from_numpy(hr).unsqueeze(0)

        return lr_tensor, hr_tensor

def true_finetune():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f" Launching True Fine-Tuning on: {device}")

    # Load the 80-Epoch Base Model weights to build upon
    model = SR_UNet_Residual().to(device)
    checkpoint = torch.load('/content/drive/MyDrive/KLA_DATA/sr_unet_residual.pth', map_location=device)

    if 'model_state' in checkpoint:
        model.load_state_dict(checkpoint['model_state'])
    else:
        model.load_state_dict(checkpoint)

    train_dataset = KLA_Dataset_Finetune('/content/dataset/train/NoisyLR', '/content/dataset/train/GT')
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)

    criterion = nn.L1Loss()

    # MICRO-OPTIMIZER: The learning rate is dropped massively to 1e-5.
    # This prevents the AI from forgetting the 80 epochs of training,
    # forcing it to only make microscopic adjustments to sharpen edges.
    optimizer = optim.Adam(model.parameters(), lr=1e-5)

    epochs = 10

    # Training Loop
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0

        pbar = tqdm(train_loader, desc=f"Fine-Tuning Epoch {epoch+1}/{epochs}")
        for lr_batch, hr_batch in pbar:
            lr_batch, hr_batch = lr_batch.to(device), hr_batch.to(device)

            optimizer.zero_grad()
            outputs = model(lr_batch)
            loss = criterion(outputs, hr_batch)

            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            pbar.set_postfix({'Loss': f"{loss.item():.5f}"})

        avg_loss = epoch_loss / len(train_loader)
        print(f"Fine-Tuning Epoch {epoch+1} Completed | Average Loss: {avg_loss:.5f}")

    # Save Ultimate Model: This is the final .pth file used in the inference script
    save_path = '/content/drive/MyDrive/KLA_DATA/sr_unet_true_ultimate.pth'
    torch.save(model.state_dict(), save_path)
    print(f"\n Final Model Saved safely to {save_path}")

true_finetune()